# Subject: ciphers
Source: /home/devuser/edu_data/datasets/TrainingDB/EncryT/secretpy/secretpy/ciphers

### File: adfgvx.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius_square import PolybiusSquare
from .columnar_transposition import ColumnarTransposition
from secretpy import alphabets as al
from itertools import starmap


class ADFGVX:
    """
    The ADFGVX Cipher
    """
    __header = u"adfgvx"
    __columnar = ColumnarTransposition()
    __alphabet = (
        u"a", u"b", u"c", u"d", u"e", u"f",
        u"g", u"h", u"i", u"j", u"k", u"l",
        u"m", u"n", u"o", u"p", u"q", u"r",
        u"s", u"t", u"u", u"v", u"w", u"x",
        u"y", u"z", u"1", u"2", u"3", u"4",
        u"5", u"6", u"7", u"8", u"9", u"0",
    )

    def encrypt(self, text, key, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English with numbers is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        alphabet = alphabet or self.__alphabet
        square = PolybiusSquare(alphabet)
        res = []
        for i, j in map(square.get_coordinates, text):
            res.append(self.__header[i])
            res.append(self.__header[j])

        res = "".join(res)
        # keyword is in english, alphabet is ENGLISH
        return self.__columnar.encrypt(res, key, al.ENGLISH)

    def decrypt(self, text, key, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        alphabet = alphabet or self.__alphabet
        # keyword is in english, alphabet is ENGLISH
        res = self.__columnar.decrypt(text, key, al.ENGLISH)
        code = []
        it = iter(res)
        try:
            for i, j in zip(it, it):
                char = i
                ii = self.__header.index(char)
                char = j
                jj = self.__header.index(char)
                coord = (ii, jj)
                code.append(coord)
        except ValueError:
            wrchar = char.encode('utf-8')
            raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")

        square = PolybiusSquare(alphabet)
        return "".join(starmap(square.get_char, code))

    def get_crypt_alphabet(self):
        return self.__header

### File: adfgx.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius_square import PolybiusSquare
from .columnar_transposition import ColumnarTransposition
from secretpy import alphabets as al
from itertools import starmap


class ADFGX:
    """
    The ADFGX Cipher
    """
    __header = u"adfgx"
    __columnar = ColumnarTransposition()

    def encrypt(self, text, key, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key is in ENGLISH
        :param alphabet: Alphabet which will be used(length=25),
                         if there is no a value, ENGLISH_SQUARE_IJ is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        square = PolybiusSquare(alphabet)
        res = []
        for i, j in map(square.get_coordinates, text):
            res.append(self.__header[i])
            res.append(self.__header[j])

        res = "".join(res)
        # keyword is in english, alphabet is ENGLISH
        return self.__columnar.encrypt(res, key, al.ENGLISH)

    def decrypt(self, text, key, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key is in ENGLISH
        :param alphabet: Alphabet which will be used(length=25),
                         if there is no a value, ENGLISH_SQUARE_IJ is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        # keyword is in english, alphabet is ENGLISH
        res = self.__columnar.decrypt(text, key, al.ENGLISH)
        code = []
        it = iter(res)
        try:
            for i, j in zip(it, it):
                char = i
                ii = self.__header.index(char)
                char = j
                jj = self.__header.index(char)
                coord = (ii, jj)
                code.append(coord)
        except ValueError:
            wrchar = char.encode('utf-8')
            raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")

        square = PolybiusSquare(alphabet)
        return "".join(starmap(square.get_char, code))

    def get_crypt_alphabet(self):
        return self.__header

### File: affine.py

In [ ]:
#!/usr/bin/python
from secretpy import alphabets as al


class Affine:
    """
    The Affine Cipher
    """

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key constisting of two parts: a and b. GCD(a, length of alphabet) = 1
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: tuple of 2 integers
        :type alphabet: string
        :return: text
        :rtype: string
        """
        a = key[0]
        b = key[1]
        subst = {c: alphabet[(i * a + b) % len(alphabet)][0] for i, letters in enumerate(alphabet) for c in letters}
        res = []
        try:
            for t in text:
                res.append(subst[t])
        except KeyError:
            wrchar = t.encode('utf-8')
            raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
        return "".join(res)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key constisting of two parts: a and b. GCD(a, length of alphabet) = 1
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: tuple of 2 integers
        :type alphabet: string
        :return: text
        :rtype: string
        """
        a = self.__inverse(key[0], len(alphabet))
        b = - key[1] * a
        inverse_key = (a, b)
        return self.encrypt(text, inverse_key, alphabet)

    def __inverse(self, a, m):
        if (m == 1 or a == 0):
            return 0
        a %= m

        mm = m
        y = 0
        x = 1
        # Extended Euclid Algorithm
        while a > 1:
            q, temp = divmod(a, mm)
            a, mm = mm, temp
            y, x = x - q * y, y
        if (x < 0):
            x += m
        return x

### File: atbash.py

In [ ]:
#!/usr/bin/python
from secretpy import alphabets as al
from .affine import Affine


class Atbash:
    """
    The Atbash Cipher
    """

    __affine = Affine()

    def encrypt(self, text, key=None, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: is not used
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: integer
        :type alphabet: string or tuple of strings
        :return: text
        :rtype: string
        """
        new_key = (-1, -1)
        return self.__affine.encrypt(text, new_key, alphabet)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: is not used
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: integer
        :type alphabet: string or tuple of strings
        :return: text
        :rtype: string
        """
        new_key = (-1, -1)
        return self.__affine.decrypt(text, new_key, alphabet)

### File: autokey.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al
from itertools import chain


class Autokey:
    """
    The Autokey Cipher
    """

    def __crypt(self, alphabet, key, text, is_encrypt):
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        res = []

        if is_encrypt == 1:
            new_key = chain(key, text)
        else:
            new_key = chain(key, res)

        for t in text:
            k = next(new_key)
            try:
                i = indexes[t]
            except ValueError:
                wrchar = t.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
            try:
                i += is_encrypt * indexes[k]
            except ValueError:
                wrchar = k.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
            i = i % len(alphabet)
            res.append(alphabet[i][0])
        return "".join(res)

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: string
        :type alphabet: string or tuple of strings
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, key, text, 1)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: string
        :type alphabet: string or tuple of strings
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, key, text, -1)

### File: bazeries.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al
from .polybius_square import PolybiusSquare
from itertools import cycle


class Bazeries:
    """
    The Bazeries Cipher
    """

    def __crypt(self, alphabet, text, key, is_encrypt=True):
        # prepare digit key
        temp = key[0]
        digitkey = []
        while temp:
            temp, rmd = divmod(temp, 10)
            digitkey.append(rmd)
        digitkey = digitkey[::-1]

        # prepare text: reversion
        i = 0
        revtext = []
        digitkey = cycle(digitkey)
        while i < len(text):
            num = next(digitkey)
            s = text[i:i + num]
            revtext.append(s[::-1])
            i += num
        revtext = u"".join(revtext)

        sq1 = PolybiusSquare(alphabet)
        # key is a number, make it a string
        sq2 = PolybiusSquare(alphabet, key[1])
        if is_encrypt:
            sq1, sq2 = sq2, sq1

        # prepare substitution from alphabet
        subst = {c: sq1.get_char(*reversed(sq2.get_coordinates(c))) for letters in alphabet for c in letters}
        # cryption
        return u"".join(subst[t] for t in revtext)

    def encrypt(self, text, key, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text, key, True)

    def decrypt(self, text, key, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text, key, False)

### File: beaufort.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from itertools import cycle
from secretpy import alphabets as al


class Beaufort:
    """
    The Beaufort Cipher
    """

    def __crypt(self, alphabet, key, text):
        # prepare alphabet for substitution
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        # prepare key
        key_indexes = (indexes[c] for c in key)
        res = []
        for k, t in zip(cycle(key_indexes), text):
            try:
                i = k - indexes[t]
                res.append(alphabet[i][0])
            except KeyError:
                wrchar = t.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
        return "".join(res)

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, key, text)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, key, text)

### File: bifid.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius_square import PolybiusSquare
from secretpy import alphabets as al


class Bifid:
    """
    The Bifid Cipher
    """

    def encrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string or tuple or list
        :return: text
        :rtype: string
        """
        key = int(key)
        if key <= 0:
            key = len(text)
        square = PolybiusSquare(alphabet)
        coords = tuple(map(square.get_coordinates, text))
        res = []
        for i in range(0, len(coords), key):
            block = coords[i:i + key]
            block = list(zip(*block))
            block = block[0] + block[1]
            for j in range(1, len(block), 2):
                res.append(square.get_char(block[j - 1], block[j]))
        return "".join(res)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string or tuple or list
        :return: text
        :rtype: string
        """
        key = int(key)
        if key <= 0:
            key = len(text)
        square = PolybiusSquare(alphabet)
        coords = tuple(map(square.get_coordinates, text))
        res = []
        for i in range(0, len(coords), key):
            block = []
            for coord in coords[i:i + key]:
                block.append(coord[0])
                block.append(coord[1])
            half = len(block) // 2
            for row, column in zip(block[:half], block[half:]):
                res.append(square.get_char(row, column))
        return "".join(res)

### File: caesar.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
import secretpy.alphabets as al
from .affine import Affine


class Caesar:
    """
    The Caesar Cipher (Shift, Additive)
    """

    __affine = Affine()

    def encrypt(self, text, key=3, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: encrypted text
        :rtype: string
        """
        new_key = (1, key)
        return self.__affine.encrypt(text, new_key, alphabet)

    def decrypt(self, text, key=3, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: decrypted text
        :rtype: string
        """
        new_key = (1, key)
        return self.__affine.decrypt(text, new_key, alphabet)

### File: caesar_progressive.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

import secretpy.alphabets as al
from itertools import chain, cycle


class CaesarProgressive:
    """
    The Caesar Progressive Cipher
    """

    def __crypt(self, alphabet, key, text):
        # prepare alphabet for substitution
        subst = {c: i for i, letters in enumerate(alphabet) for c in letters}
        res = []
        for t, k in zip(text, cycle(key)):
            try:
                i = subst[t] - k
            except KeyError:
                wrchar = t.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
            res.append(alphabet[i][0])
        return "".join(res)

    def encrypt(self, text, key=3, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: encrypted text
        :rtype: string
        """
        new_key = -key % len(alphabet)
        new_key = chain(range(new_key, -1, -1), range(len(alphabet) - 1, new_key, -1))
        return self.__crypt(alphabet, new_key, text)

    def decrypt(self, text, key=3, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: decrypted text
        :rtype: string
        """
        new_key = key % len(alphabet)
        new_key = chain(range(new_key, len(alphabet)), range(new_key))
        return self.__crypt(alphabet, new_key, text)

### File: chao.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-


class Chao:
    """
    The Chaocipher
    """

    def __encDec(self, text, isEncrypt, tp_alphabet, tc_alphabet):
        ret = ''
        for c in text:
            try:
                if isEncrypt:
                    i = tp_alphabet.index(c)
                    ret += tc_alphabet[i]
                else:
                    i = tc_alphabet.index(c)
                    ret += tp_alphabet[i]
            except ValueError:
                wrchar = c.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
            tc_alphabet = self.permuteAlphabet(tc_alphabet, i, True)
            tp_alphabet = self.permuteAlphabet(tp_alphabet, i, False)
        return ret

    def encrypt(self, text, key, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: encrypted text
        :rtype: string
        """
        return self.__encDec(text, True, alphabet, key)

    def decrypt(self, text, key, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: decrypted text
        :rtype: string
        """
        return self.__encDec(text, False, alphabet, key)

    def permuteAlphabet(self, alphabet, i, isCrypt):
        alphabet = alphabet[i:] + alphabet[:i]
        nadir = len(alphabet) / 2
        if isCrypt:
            alphabet = alphabet[0] + alphabet[2:int(nadir) + 1] + alphabet[1] + alphabet[int(nadir) + 1:]
        else:
            alphabet = alphabet[1:] + alphabet[0]
            alphabet = alphabet[:2] + alphabet[3:int(nadir) + 1] + alphabet[2] + alphabet[int(nadir) + 1:]
        return alphabet

### File: columnar_transposition.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al


class ColumnarTransposition:
    """
    The Columnar Transposition Cipher
    """

    def __keyorder(self, alphabet, key):
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        new_key = map(lambda x: indexes[x], key)
        return [i for i, _ in sorted(enumerate(new_key), key=lambda x: x[1])]

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        keyorder = self.__keyorder(alphabet, key)
        return u"".join(text[i::len(key)] for i in keyorder)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        keyorder = self.__keyorder(alphabet, key)
        rows, rmd = divmod(len(text), len(key))

        p = []
        acc = 0
        for i in keyorder:
            p.append(acc)
            acc += rows
            if i < rmd:
                acc += 1

        roworder = [0] * len(key)
        for i, j in enumerate(keyorder):
            roworder[j] = p[i]

        res = [text[i + row] for row in range(rows) for i in roworder]
        # add tail(remainder) to the end of result
        for i in roworder[:rmd]:
            res.append(text[i + rows])
        return u"".join(res)

### File: enigma.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
import secretpy.alphabets as al


class Enigma:
    """
    The Enigma Cipher
    """

    class Rotor:
        def __init__(self, key, alphabet, turnover, position=0, ringsetting=0):
            if isinstance(position, str):
                self.position = alphabet.index(position)
            else:
                self.position = position
            if isinstance(ringsetting, str):
                self.ring = alphabet.index(ringsetting)
            else:
                self.ring = ringsetting
            self.key = key  # the same length as alphabet
            self.alphabet = alphabet
            self.next_rotor = None
            self.turnover = turnover

        def encrypt_char(self, char):
            i = (self.alphabet.index(char) + self.position - self.ring) % len(self.alphabet)
            c = self.key[i]
            i = (self.alphabet.index(c) - self.position + self.ring) % len(self.alphabet)
            return self.alphabet[i]

        def inverse_encrypt(self, char):
            i = (self.alphabet.index(char) + self.position - self.ring) % len(self.alphabet)
            c = self.alphabet[i]
            i = (self.key.index(c) - self.position + self.ring) % len(self.alphabet)
            return self.alphabet[i]

        def rotate(self):
            if self.alphabet[self.position][0] in self.turnover:
                if self.next_rotor:
                    self.next_rotor.rotate()
            self.position += 1
            self.position %= len(self.alphabet)

        def set_next_rotor(self, rotor):
            self.next_rotor = rotor

        def get_position(self):
            return self.alphabet[self.position]

        # end of class Rotor

    def encrypt(self, text, key=None, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: encrypted text
        :rtype: string
        """
        if not isinstance(key, dict):
            key = {
                'reflector': 'B',
                'rotor_offsets': ('a', 'a', 'a'),  # Grundstellung
                'ring_offsets': ('a', 'a', 'a'),   # Ringstellung
                'rotor_order': (1, 2, 3),
                'steckers': []
            }

        exist_rotors = (
            {'name': 'ENIGMA_I_I', 'wiring': "ekmflgdqvzntowyhxuspaibrcj", 'overturn': 'q'},
            {'name': 'ENIGMA_I_II', 'wiring': "ajdksiruxblhwtmcqgznpyfvoe", 'overturn': 'e'},
            {'name': 'ENIGMA_I_III', 'wiring': "bdfhjlcprtxvznyeiwgakmusqo", 'overturn': 'v'},
            {'name': 'M3_ARMY_IV', 'wiring': "esovpzjayquirhxlnftgkdcmwb", 'overturn': 'j'},
            {'name': 'M3_ARMY_V', 'wiring': "vzbrgityupsdnhlxawmjqofeck", 'overturn': 'z'},
            {'name': 'M3_M4_NAVAL_VI', 'wiring': "jpgvoumfyqbenhzrdkasxlictw", 'overturn': 'zm'},
            {'name': 'M3_M4_NAVAL_VII', 'wiring': "nzjhgrcxmyswboufaivlpekqdt", 'overturn': 'zm'},
            {'name': 'M3_M4_NAVAL_VIII', 'wiring': "fkqhtlxocbjspdzramewniuygv", 'overturn': 'zm'},
        )

        exist_reflectors = {
            'A': "ejmzalyxvbwfcrquontspikhgd",
            'B': "yruhqsldpxngokmiebfzcwvjat",
            'C': "fvpjiaoyedrzxwgctkuqsbnmhl",
        }

        plugboard = {c: c for c in alphabet}
        for s1, s2 in key['steckers']:
            plugboard[s1] = s2
            plugboard[s2] = s1

        # set reflector, it behaves as a rotor without rotation
        rotors = [self.Rotor(exist_reflectors[key['reflector']], alphabet, 0, 0, 0)]

        # set other rotors
        for i, ri in enumerate(key['rotor_order']):
            r = exist_rotors[ri - 1]
            rotor = self.Rotor(r['wiring'], alphabet, r['overturn'], key['rotor_offsets'][i], key['ring_offsets'][i])
            rotors.append(rotor)

        # connect rotors
        for i in range(len(rotors) - 1, 1, -1):
            rotors[i].set_next_rotor(rotors[i - 1])

        res = []
        for c in text:
            # rotate before send a char
            rotors[-1].rotate()
            c = plugboard[c]
            # we go from right to left
            for r in reversed(rotors):
                c = r.encrypt_char(c)
            it = iter(rotors)
            next(it)  # omit reflector
            # and then go back
            for r in it:
                c = r.inverse_encrypt(c)
            c = plugboard[c]
            res.append(c)
        return "".join(res)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: decrypted text
        :rtype: string
        """
        return self.encrypt(text, key, alphabet)

### File: four_square.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius_square import PolybiusSquare
from secretpy import alphabets as al


class FourSquare:
    """
    The Four-Square Cipher
    """

    def __crypt(self, alphabet, text, key, is_encrypt):
        square01 = PolybiusSquare(alphabet, key[0])
        square10 = PolybiusSquare(alphabet, key[1])
        square = PolybiusSquare(alphabet)
        square00 = square
        square11 = square

        res = []
        if is_encrypt:
            square01, square10, square00, square11 = square00, square11, square01, square10

        # text encryption
        it = iter(text)
        while True:
            try:
                even = next(it)
            except StopIteration:
                break
            try:
                odd = next(it)
            except StopIteration:
                # add the last letter in the alphabet
                odd = alphabet[-1][0]
            row00, column00 = square01.get_coordinates(even)
            row11, column11 = square10.get_coordinates(odd)
            res.append(square00.get_char(row00, column11))
            res.append(square11.get_char(row11, column00))
        return "".join(res)

    def encrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: tuple of two strings
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text, key, True)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: tuple of two strings
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text, key, False)

### File: gronsfeld.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from itertools import cycle
from secretpy import alphabets as al


class Gronsfeld:
    """
    The Gronsfeld Cipher
    """

    def __crypt(self, alphabet, key, text):
        # prepare alphabet for substitution
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        # prepare key
        res = []
        for keyi, char in zip(cycle(key), text):
            try:
                i = indexes[char] - keyi
                res.append(alphabet[i][0])
            except KeyError:
                wrchar = char.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
        return "".join(res)

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: tuple of integers
        :type alphabet: string
        :return: text
        :rtype: string
        """
        # sanitize and invert the key
        new_key = (-i % len(alphabet) for i in key)
        return self.__crypt(alphabet, new_key, text)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: tuple of integers
        :type alphabet: string
        :return: text
        :rtype: string
        """
        # sanitize the key
        new_key = (i % len(alphabet) for i in key)
        return self.__crypt(alphabet, new_key, text)

### File: keyword.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from collections import OrderedDict
from secretpy import alphabets as al


class Keyword:
    """
    The Keyword Cipher
    """

    def __crypt(self, alphabet, key, text, is_encrypt):
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        # remove duplicates
        keyi = OrderedDict.fromkeys(indexes[char] for char in key)
        new_key = [alphabet[i] for i in keyi]
        for i, a in enumerate(alphabet):
            if i not in keyi:
                new_key.append(a[0])

        if is_encrypt:
            subst = {c: new_key[i] for c, i in indexes.items()}
        else:
            subst = {c: alphabet[i][0] for i, letters in enumerate(new_key) for c in letters}
        # do encryption
        res = []
        for t in text:
            try:
                res.append(subst[t])
            except KeyError:
                wrchar = t.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
        return "".join(res)

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, key, text, True)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, key, text, False)

### File: myszkowski_transposition.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from itertools import repeat
from secretpy import alphabets as al


class MyszkowskiTransposition:
    """
    The Myszkowski Transposition Cipher
    """
    def __keycols(self, key, alphabet):
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        chars = [indexes[char] for char in key]
        chars = sorted(enumerate(chars), key=lambda x: x[1])
        cols = [[chars[0][0]]]

        for i in range(1, len(key)):
            if chars[i - 1][1] == chars[i][1]:
                cols[-1].append(chars[i][0])
            else:
                cols.append([chars[i][0]])
        return cols

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        res = []
        keyorder = self.__keycols(key, alphabet)
        for columns in keyorder:
            if len(columns) == 1:
                res.append(text[columns[0]::len(key)])
            else:
                # use zip_longest in python3
                iterators = [iter(text[i::len(key)]) for i in columns]
                it_count = len(iterators)
                while it_count:
                    for i, it in enumerate(iterators):
                        try:
                            res.append(next(it))
                        except StopIteration:
                            it_count -= 1
                            iterators[i] = repeat('')
        return u"".join(res)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used, if there is no a value,
                         English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        cols = self.__keycols(key, alphabet)

        lines, rmd = divmod(len(text), len(key))
        lines += rmd > 0

        columns = [[] for _ in range(len(key))]
        texti = iter(text)
        # fill columns
        for col in cols:
            ccol = col
            for _ in range(lines):
                for i, value in enumerate(ccol):
                    if value < len(text):
                        columns[col[i]].append(next(texti))
                ccol = [i + len(key) for i in ccol]

        # read result from columns
        res = []
        iterators = [iter(column) for column in columns]
        loop = True
        while loop:
            for it in iterators:
                try:
                    res.append(next(it))
                except StopIteration:
                    loop = False
                    break
        return u"".join(res)

### File: nihilist.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius import Polybius
from secretpy import alphabets as al


class Nihilist:
    """
    The Nihilist Cipher
    """

    __polybius = Polybius()

    def encrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        code = self.__polybius.encrypt(text, alphabet=alphabet)
        enc = ""
        for i in range(0, len(code), 2):
            char = self.__polybius.encrypt(key[(i >> 1) % len(key)],
                                           alphabet=alphabet)
            enc += str(int(code[i:i + 2]) + int(char)) + " "
        return enc.rstrip()

    def decrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        code = text.split(' ')
        code = list(map(int, code))
        dec = ""
        for i in range(0, len(code)):
            char = self.__polybius.encrypt(key[i % len(key)],
                                           alphabet=alphabet)
            pair = str(code[i] - int(char))
            dec += self.__polybius.decrypt(pair, alphabet=alphabet)
        return dec

    def get_crypt_alphabet(self):
        return al.DECIMAL + " "

### File: playfair.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius_square import PolybiusSquare
from secretpy import alphabets as al


class Playfair:
    """
    The Playfair Cipher
    """

    # encrypt or decrypt two letters
    def __crypt(self, a, b, square, is_encrypt):
        cols = square.get_columns()
        rows = square.get_rows()

        arow, acolumn = square.get_coordinates(a)
        brow, bcolumn = square.get_coordinates(b)

        if arow == brow:
            acolumn = (acolumn + is_encrypt) % cols
            bcolumn = (bcolumn + is_encrypt) % cols
        elif acolumn == bcolumn:
            arow += is_encrypt
            brow += is_encrypt
            if arow >= rows:
                arow = 0
            if brow >= rows:
                brow = 0
        else:
            acolumn, bcolumn = bcolumn, acolumn
        a = square.get_char(arow, acolumn)
        b = square.get_char(brow, bcolumn)
        return a + b

    def encrypt(self, text, key="", alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, ENGLISH_SQUARE_IJ is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        insert_char = 'x'
        square = PolybiusSquare(alphabet, key)

        # prepare text
        txt = []
        i = 1
        while i < len(text):
            txt.append(text[i - 1])
            if text[i - 1] == text[i]:
                txt.append(insert_char)
                i += 1
            else:
                txt.append(text[i])
                i += 2
        # add the last character
        if i == len(text):
            txt.append(text[i - 1])
            txt.append(insert_char)

        return "".join(self.__crypt(txt[i - 1], txt[i], square, 1) for i in range(1, len(txt), 2))

    def decrypt(self, text, key="", alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, ENGLISH_SQUARE_IJ is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        insert_char = 'x'
        square = PolybiusSquare(alphabet, key)

        res = [self.__crypt(text[i - 1], text[i], square, -1) for i in range(1, len(text), 2)]
        # remove the insert character
        for i in range(1, len(res)):
            if res[i - 1][0] == res[i][0] and res[i - 1][1] == insert_char:
                res[i - 1] = res[i - 1][0]
        # check the last character
        if res[-1][1] == insert_char:
            res[-1] = res[-1][0]

        return "".join(res)

### File: polybius.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from itertools import starmap
from .polybius_square import PolybiusSquare
import secretpy.alphabets as al


class Polybius:
    """
    The Polybius Cipher
    """

    def encrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        square = PolybiusSquare(alphabet, key)
        res = []
        for row, column in map(square.get_coordinates, text):
            res.append(str(row + 1))
            res.append(str(column + 1))
        return "".join(res)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        square = PolybiusSquare(alphabet, key)
        it = iter(map(lambda t: int(t) - 1, text))
        return "".join(starmap(square.get_char, zip(it, it)))

    def get_crypt_alphabet(self):
        return al.DECIMAL

### File: polybius_square.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

import math
from collections import OrderedDict
from secretpy import alphabets as al


class PolybiusSquare:
    """
    PolybiusSquare. It's used by many classical ciphers
    """

    def __init__(self, alphabet=al.ENGLISH_SQUARE_IJ, key=None):
        self.__side = int(math.ceil(math.sqrt(len(alphabet))))
        # prepare alphabet for substitution
        if key:
            indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
            # remove duplicates
            keyi = OrderedDict.fromkeys(indexes[char] for char in key)
            self.__alphabet = [alphabet[i] for i in keyi]

            for i, a in enumerate(alphabet):
                if i not in keyi:
                    self.__alphabet.append(a)
        else:
            self.__alphabet = alphabet
        self.coords = {c: divmod(i, self.__side) for i, letters in enumerate(self.__alphabet) for c in letters}

    def get_coordinates(self, char):
        return self.coords[char]

    def get_char(self, row, column):
        return self.__alphabet[row * self.__side + column][0]

    def get_columns(self):
        return self.__side

    def get_rows(self):
        return len(self.__alphabet) // self.__side

### File: porta.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from itertools import cycle
from secretpy import alphabets as al


class Porta:
    """
    The Porta Cipher
    """

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        # prepare alphabet for substitution
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        # prepare key
        key_indexes = (indexes[c] >> 1 for c in key)
        half = len(alphabet) >> 1
        res = []
        for t, k in zip(text, cycle(key_indexes)):
            try:
                texti = indexes[t]
            except ValueError:
                wrchar = t.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
            if texti < half:
                i = (texti + k) % half + half
            else:
                i = (texti - k) % half
            res.append(alphabet[i][0])
        return u"".join(res)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.encrypt(text, key, alphabet)

### File: rot13.py

In [ ]:
#!/usr/bin/python

from .caesar import Caesar
import secretpy.alphabets as al


class Rot13:
    """
    The Rot13 Cipher (Half)
    """
    __caesar = Caesar()

    def __crypt(self, alphabet, text):
        alph = alphabet
        key = len(alph) >> 1
        # if number of letters in the alphabet is odd
        if len(alph) & 1:
            alph += alph[key]
            key += 1
        return self.__caesar.encrypt(text, key, alph)

    def encrypt(self, text, key=None, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: not used
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: not used
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text)

### File: rot18.py

In [ ]:
#!/usr/bin/python

from .rot13 import Rot13
import secretpy.alphabets as al


class Rot18:
    """
    The Rot18 Cipher
    """
    __rot13 = Rot13()

    def __init__(self):
        alphabet = al.ENGLISH
        half = len(alphabet) >> 1
        self.__alphabet = alphabet[:half] + al.DECIMAL[:5] + alphabet[half:] + al.DECIMAL[5:]

    def __crypt(self, text, alphabet):
        return self.__rot13.encrypt(text, alphabet=self.__alphabet)

    def encrypt(self, text, key=None, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: is not used
        :param alphabet: is not used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(text, self.__alphabet)

    def decrypt(self, text, key=None, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: is not used
        :param alphabet: is not used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(text, self.__alphabet)

    def get_fixed_alphabet(self):
        return self.__alphabet

### File: rot47.py

In [ ]:
#!/usr/bin/python

from .caesar import Caesar


class Rot47:
    """
    The Rot47 Cipher
    """

    __caesar = Caesar()
    __alphabet = "".join(chr(asc) for asc in range(33, 33 + 47 * 2))

    def __crypt(self, text):
        return self.__caesar.encrypt(text, 47, self.__alphabet)

    def encrypt(self, text, key=None, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(text)

    def decrypt(self, text, key=None, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(text)

    def get_fixed_alphabet(self):
        return self.__alphabet

### File: rot5.py

In [ ]:
#!/usr/bin/python

from .caesar import Caesar
import secretpy.alphabets as al


class Rot5:
    """
    The Rot5 Cipher
    """
    __caesar = Caesar()

    def encrypt(self, text, key=None, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__caesar.encrypt(text, 5, al.DECIMAL)

    def decrypt(self, text, key=None, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__caesar.encrypt(text, 5, al.DECIMAL)

    def get_fixed_alphabet(self):
        return al.DECIMAL

### File: scytale.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-


class Scytale:
    """
    The Scytale Cipher
    """

    def encrypt(self, text, key, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key - Number of windings
        :param alphabet: Alphabet is not used
        :type text: string
        :type key: integer
        :return: encrypted text
        :rtype: string
        """
        # key is columns
        return "".join(text[i::key] for i in range(key))

    def decrypt(self, text, key, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key - Number of windings
        :param alphabet: Alphabet is not used
        :type text: string
        :type key: integer
        :return: decrypted text
        :rtype: string
        """
        full_rows, rmd = divmod(len(text), key)
        rows = full_rows + (rmd > 0)
        middle = rows * rmd
        res = []
        for i in range(full_rows):
            res.append(text[i:middle:rows])
            res.append(text[middle + i::full_rows])
        res.append(text[full_rows:middle:rows])
        return "".join(res)

### File: simplesubstitution.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al


class SimpleSubstitution:
    """
    The Simple Substitution Cipher
    """
    def __crypt(self, subst, text):
        res = []
        for c in text:
            try:
                res.append(subst[c])
            except KeyError:
                wrchar = c.encode('utf-8')
                raise Exception("Can't find char '" + wrchar + "' of text in alphabet!")
        return "".join(res)

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        if len(alphabet) != len(key):
            raise Exception("Lengths of alphabet and key should be the same")
        # prepare alphabet for substitution
        subst = {a: k[0] for k, letters in zip(key, alphabet) for a in letters}
        return self.__crypt(subst, text)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        if len(alphabet) != len(key):
            raise Exception("Lengths of alphabet and key should be the same")
        # prepare alphabet for substitution
        subst = {k[0]: a[0] for k, a in zip(key, alphabet)}
        return self.__crypt(subst, text)

### File: spiral.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from itertools import cycle


class Spiral:
    """
    The Spiral Transposition Cipher

    It's a variant of the Route Cipher
    Spiral inwards, clockwise, starting at the top right.

    https://en.wikipedia.org/wiki/Transposition_cipher#Route_cipher
    """
    def __crypt(self, text, key, is_encrypt):
        columns, rmd = divmod(len(text), key)

        # init
        top = 0
        left = key - 1
        bottom = key * (columns + (rmd > 0)) - 1

        if rmd:
            right = len(text) - rmd
        else:
            right = len(text) - key
        i = right
        right -= key

        if is_encrypt:
            res = []
        else:
            ti = 0
            res = [None] * len(text)

        # i = from, j = to

        # top right to bottom right
        j = bottom
        if i == j:
            if is_encrypt:
                res.append(text[i])
            else:
                res[i] = text[ti]
            return u"".join(res)
        temp = j
        if temp > len(text):
            temp = len(text)
        for p in range(i, temp):
            if is_encrypt:
                res.append(text[p])
            else:
                res[p] = text[ti]
                ti += 1
        # change limit
        bottom -= 1 + key

        # bottom right to bottom left
        i = j
        if rmd:  # additional condition
            i -= key
        if i < 0:  # additional condition
            return u"".join(res)
        j = left
        if i == j and rmd == 0:
            if is_encrypt:
                res.append(text[i])
            else:
                res[i] = text[ti]
            return u"".join(res)
        for p in range(i, j, -key):
            if is_encrypt:
                res.append(text[p])
            else:
                res[p] = text[ti]
                ti += 1
        # change limit
        left += key - 1

        # ([limit, change_limit, delta], ...)
        steps = ([top, key + 1, -1], [right, 1 - key, key], [bottom, -key - 1, 1], [left, key - 1, -key])

        # bottom left to top left
        # top left to top right
        # top right to bottom right
        # bottom right to bottom left
        for step in cycle(steps):
            i = j
            j = step[0]
            step[0] += step[1]
            if i == j:
                if is_encrypt:
                    res.append(text[i])
                else:
                    res[i] = text[ti]
                break
            for p in range(i, j, step[2]):
                if is_encrypt:
                    res.append(text[p])
                else:
                    res[p] = text[ti]
                    ti += 1
        return u"".join(res)

    def encrypt(self, text, key, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Number of rows
        :param alphabet: Alphabet is not used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(text, key, True)

    def decrypt(self, text, key, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Number of rows
        :param alphabet: Alphabet is not used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(text, key, False)

### File: three_square.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al
from .polybius_square import PolybiusSquare
import random


class ThreeSquare:
    """
    The Three Square Cipher
    """

    def encrypt(self, text, key, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: tuple of 3 strings
        :type alphabet: string
        :return: text
        :rtype: string
        """
        square1 = PolybiusSquare(alphabet, key[0])
        square2 = PolybiusSquare(alphabet, key[1])
        square3 = PolybiusSquare(alphabet, key[2])

        res = []
        it = iter(text)
        rows = square1.get_rows()
        cols = square2.get_columns()
        while True:
            try:
                t = next(it)
            except StopIteration:
                break
            row1, column1 = square1.get_coordinates(t)
            try:
                t = next(it)
            except StopIteration:
                t = alphabet[-1][0]
            row2, column2 = square2.get_coordinates(t)

            i = random.randrange(rows)
            left = square1.get_char(i, column1)

            middle = square3.get_char(row1, column2)

            i = random.randrange(cols)
            right = square2.get_char(row2, i)

            res.append(left)
            res.append(middle)
            res.append(right)
        return u"".join(res)

    def decrypt(self, text, key, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: tuple of 3 strings
        :type alphabet: string
        :return: text
        :rtype: string
        """
        square1 = PolybiusSquare(alphabet, key[0])
        square2 = PolybiusSquare(alphabet, key[1])
        square3 = PolybiusSquare(alphabet, key[2])

        res = []
        it = iter(text)
        for t0, t1, t2 in zip(it, it, it):
            col1 = square1.get_coordinates(t0)[1]
            row3, col3 = square3.get_coordinates(t1)
            row2 = square2.get_coordinates(t2)[0]
            res.append(square1.get_char(row3, col1))
            res.append(square2.get_char(row2, col3))
        return u"".join(res)

### File: trifid.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from secretpy import alphabets as al
from itertools import product


class Trifid:
    """
    The Trifid Cipher
    """
    alphabet = al.ENGLISH + "."

    def __code(self, text, alphabet):
        # prepare coordinates for each character in the alphabet
        # coord : (square, row, column)
        coords = {c: coord for i, coord in enumerate(product(range(3), repeat=3)) for c in alphabet[i]}
        code = []
        for char in text:
            code.extend(coords[char])
        return code

    def __decode(self, code, alphabet):
        text = []
        size = 3
        for i in range(0, len(code), size):
            index = code[i]  # square
            index = index * size + code[i + 1]  # row
            index = index * size + code[i + 2]  # column
            text.append(alphabet[index][0])
        return "".join(text)

    def encrypt(self, text, key=None, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English with '.' is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        alphabet = alphabet or self.alphabet
        assert len(alphabet) == 27, "Length of the alphabet should be 27"

        size = 3
        key = int(key)
        if key <= 0:
            key = len(text)
        code = self.__code(text, alphabet)

        code0 = []
        for j in range(0, len(text) * size, size * key):
            for i in range(size):
                for item in code[j + i:j + size * key:size]:
                    code0.append(item)

        return self.__decode(code0, alphabet)

    def decrypt(self, text, key=None, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English with '.' is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        alphabet = alphabet or self.alphabet
        assert len(alphabet) == 27, "Length of the alphabet should be 27"

        key = int(key)
        if key <= 0:
            key = len(text)
        code = self.__code(text, alphabet)

        size = 3
        res = []
        period = size * key
        for i in range(0, len(code), period):
            block = code[i:i + period]
            third = len(block) // size
            # coord : (square, row, column)
            for coord in zip(block[:third], block[third:2 * third], block[2 * third:]):
                res.extend(coord)
        return self.__decode(res, alphabet)

### File: two_square.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-

from .polybius_square import PolybiusSquare
from secretpy import alphabets as al


class TwoSquare:
    """
    The Two-Square Cipher, also called Double Playfair
    """

    def __crypt(self, alphabet, text, key):
        square1 = PolybiusSquare(alphabet, key[0])
        square2 = PolybiusSquare(alphabet, key[1])

        # text encryption
        res = []
        it = iter(text)
        while True:
            try:
                even = next(it)
            except StopIteration:
                break
            try:
                odd = next(it)
            except StopIteration:
                # add the last letter in the alphabet
                odd = alphabet[-1][0]
            row1, column1 = square1.get_coordinates(even)
            row2, column2 = square2.get_coordinates(odd)

            if column1 == column2:
                row1, row2 = row2, row1

            res.append(square1.get_char(row1, column2))
            res.append(square2.get_char(row2, column1))

        return "".join(res)

    def encrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text, key)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        return self.__crypt(alphabet, text, key)

### File: vic.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al
from itertools import cycle


class Vic:
    """
    The Vic Cipher
    """

    def __crypt(self, alphabet, text, key):
        columns = []
        subst = {}
        width = 10
        # prepare substitution from alphabet
        for i, value in enumerate(alphabet):
            if value == "":
                columns.append(i)
            elif i < width:
                subst[value] = (i,)
            else:
                row, col = divmod(i, width)
                subst[value] = (columns[row - 1], col)

        # encode chars to numbers
        code = []
        for t in text:
            code.extend(subst[t])

        res = []
        row = 0
        for c, k in zip(code, cycle(key)):
            # apply the key
            i = (c + k) % width
            # encode numbers to chars
            if row == 0 and (i in columns):
                row = columns.index(i) + 1
            else:
                res.append(alphabet[row * width + i][0])
                row = 0
        return u"".join(res)

    def encrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: number string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        # prepare key
        new_key = (int(i) for i in key)
        return self.__crypt(alphabet, text, new_key)

    def decrypt(self, text, key=None, alphabet=al.ENGLISH_SQUARE_IJ):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: number string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        width = 10
        # invert key
        new_key = (width - int(i) for i in key)
        return self.__crypt(alphabet, text, new_key)

### File: vigenere.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-
from secretpy import alphabets as al
from .gronsfeld import Gronsfeld


class Vigenere:
    """
    The Vigenere Cipher
    """
    __gronsfeld = Gronsfeld()

    def __crypt(self, alphabet, key):
        # prepare alphabet for substitution
        indexes = {c: i for i, letters in enumerate(alphabet) for c in letters}
        # prepare key
        return (indexes[c] for c in key)

    def encrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        new_key = self.__crypt(alphabet, key)
        return self.__gronsfeld.encrypt(text, new_key, alphabet)

    def decrypt(self, text, key, alphabet=al.ENGLISH):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: Alphabet which will be used,
                         if there is no a value, English is used
        :type text: string
        :type key: string
        :type alphabet: string
        :return: text
        :rtype: string
        """
        new_key = self.__crypt(alphabet, key)
        return self.__gronsfeld.decrypt(text, new_key, alphabet)

### File: zigzag.py

In [ ]:
#!/usr/bin/python
# -*- encoding: utf-8 -*-


class Zigzag:
    """
    The Zigzag Cipher (Rail-Fence)
    """

    def encrypt(self, text, key, alphabet=None):
        """
        Encryption method

        :param text: Text to encrypt
        :param key: Encryption key
        :param alphabet: unused
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        if key <= 0:
            return
        key0 = key - 1
        step = key0 << 1

        # the first row
        crypted = [text[::step]]

        # next rows
        textlen = len(text)
        for row in range(1, key0):
            right = step - row
            for left in range(row, textlen, step):
                crypted.append(text[left])
                if right < textlen:
                    crypted.append(text[right])
                right += step
        # the last row
        crypted.append(text[key0::step])
        return "".join(crypted)

    def decrypt(self, text, key, alphabet=None):
        """
        Decryption method

        :param text: Text to decrypt
        :param key: Decryption key
        :param alphabet: unused
        :type text: string
        :type key: integer
        :type alphabet: string
        :return: text
        :rtype: string
        """
        step = (key - 1) << 1
        textlen = len(text)
        decrypted = [None] * textlen

        # first row
        i = 0
        for left in range(0, textlen, step):
            decrypted[left] = text[i]
            i += 1

        # next rows
        for row in range(1, key):
            for left in range(row, textlen, step):
                decrypted[left] = text[i]
                i += 1
                right = left + step - (row << 1)
                if right < textlen and right != left:
                    decrypted[right] = text[i]
                    i += 1

        return "".join(decrypted)